# Matching Names and ORCIDs
Given a list of author names we can search for ORCID metadata using getORCIDS.py. This retrieves data from ORCID profiles including: other names, employment, education, works, peer-reviews, funding and email. In order for these retrievals to work, the ORCID profile data must 1) exist in the profile and 2) be open to the public. If these data are available, it is not unusual for there to be multiple matches to many names, after all, that is why ORCIDs exist.

This notebook explores a possible rubric fort describing the goodness-of-fit of a match between a name and an ORCID. It is, of course, not fool proof, but it may be helpful.

In [ ]:
import pandas as pd
from   tabulate import tabulate                     # makes pretty tables in many formats from dataframes
from   IPython.display import display, Markdown     # allows computation results to be displayed in markdown

In [ ]:
#
# Data and metadata related to these papers are in a public git repository
# https://github.com/tedhabermann/MooreaStudentPapers
# The index of the papers (a spreadsheet) is in this git repository as a csv file
# read the index of student papers from git repository
#
url = 'https://raw.githubusercontent.com/tedhabermann/MooreaStudentPapers/refs/heads/main/data/index_to_moorea_class_projects_1992-2022.csv'
index_df = pd.read_csv(url)

# Display the sheet names
print(f"{len(index_df)} rows in the index dataframe")


In [ ]:
#
# Read all sheets from the ORCID metadata files in the git repository
# These are csv files spilt out from the ORCID metadata spreadsheet (xlsx) that is in the git repository
#
orcid_metadata_l = ['counts', 'employment', 'education', 'works', 'funding', 'email']
orcid_dfs = {}
for sheet in orcid_metadata_l:
    url = f"https://raw.githubusercontent.com/tedhabermann/MooreaStudentPapers/refs/heads/main/data/orcidMetadata_{sheet}__20260303_14.csv"
    print(f"Reading {sheet} sheet from ORCID metadata file at {url}")
    orcid_dfs[sheet] = pd.read_csv(url)
    print(f"{len(orcid_dfs[sheet])} rows in the {sheet} dataframe")

# Display the sheet names
print(f"Sheets found: {list(orcid_dfs.keys())}")

In [ ]:
# Read the index of student papers, which includes author names and publication years, into a dataframe
index_file = '/Users/metadatagamechanger/ProjectsAndPlans/FieldStations/MooreaStudentPapers/index_to_moorea_class_projects_1992-2022.xlsx'
index_df = pd.read_excel(index_file, sheet_name='MooreaClass-Export.txt', engine='openpyxl')

# Display the sheet names
print(f"{len(index_df)} rows in the index dataframe")

In [ ]:
#
# Read all sheets from the ORCID metadata file (xlsx) into a dictionary of dataframes
#
orcid_metadata_file = '/Users/metadatagamechanger/Repositories/MooreaStudentPapers/data/orcidMetadata__20260303_14.xlsx'
orcid_dfs = pd.read_excel(orcid_metadata_file, sheet_name=None)

# Display the sheet names
print(f"Sheets found: {list(orcid_dfs.keys())}")

## Match Counts
An ORCID search for a name can return any number of matches, i.e. one for each person with that name that has an ORCID. The difficulty of picking the right ORCID increases as the number of matches increases. The counts dataframe in the ORCID output has the input names in a column named 'input'. The number of times these mnames occur is the number of matches for the name. Add these counts to the counts dataframe as the matchCounts column.

In [ ]:
#
# Create a series of counts of the number of matches for each input name, and add it to the dataframe
#
matchCounts              = counts_df[counts_df['orcid'] != 'Not Found']['input'].value_counts()     # a series with the number of matches for each input name
counts_df['matchCounts'] = counts_df['input'].map(matchCounts).fillna(0).astype(int)                # add the match counts to the dataframe, filling in 0 for names with no matches
counts_df.fillna('', inplace=True)                                                                  # replace NaN with empty string for better display
counts_df.head(20)                                                                                  # display the first 20 rows of the dataframe with match counts

In [ ]:
studyYear = 2004                         # pick a year to examine
studyYear_df = index_df[index_df['Pub Year'] == studyYear]
print(f"{len(studyYear_df)} authors in {studyYear}")


In [ ]:
author_l = list(studyYear_df['Authors, Primary'].values)                               # display the keys of the 'Authors, Primary' column for the selected year
authorORCIDMetadata_df = counts_df[counts_df['input'].isin(author_l)]         # get the dataframe for the 'authorORCIDMetadata' sheet
#display(Markdown(tabulate(authorORCIDMetadata_df.matchCounts.value_counts().reset_index(), headers=['Match Count', 'Frequency'], tablefmt='pipe', floatfmt='.0f', showindex=False)))
display(Markdown(f'**{len(studyYear_df)} authors in {studyYear}**'))
display(Markdown(tabulate(authorORCIDMetadata_df.sort_values(by='matchCounts'), headers=list(authorORCIDMetadata_df.columns), tablefmt='pipe', floatfmt='.0f', showindex=False)))


In [ ]:
element_l = ['email','education', 'employment', 'works','funding']

input = 'Kinney,Michael J.'
orcid_l = list(authorORCIDMetadata_df[authorORCIDMetadata_df['input']==input]['orcid'].values)

if orcid_l[0] == 'Not Found':
    print(f'No ORCID was found for {input}')
else:
    if len(orcid_l) == 1:
        display(Markdown(f'{input} has 1 ORCID match: [{orcid_l[0]}](https://commons.datacite.org/orcid.org/{orcid_l[0]})'))
    else:
        print(f'{input} has {len(orcid_l)} ORCID matchs: ', ', '.join(orcid_l))
    for element in element_l:
        data = orcid_dfs[element]
        df = data[data['orcid'].isin(orcid_l)]
        if len(df) > 0:
            df = df.fillna('')
            display(Markdown(f'**{element}: ({len(df)})**'))
            display(Markdown(tabulate(df, headers=list(df.columns), tablefmt='pipe', floatfmt='.0f', showindex=False)))
        else:
            display(Markdown(f'{input} has no {element} metadata.'))

In [ ]:
oneMatch_df = counts_df[counts_df['matchCounts'] == 1]

In [ ]:
#oneMatch_Counts_df = oneMatch_df.merge(counts_df, on='input', how='left')
#oneMatch_Counts_df = oneMatch_Counts_df.fillna('')

In [ ]:
display(Markdown(f'**{len(oneMatch_df)} People with one ORCID match**'))
display(Markdown(tabulate(oneMatch_df.sort_values(by='input'), headers=list(oneMatch_df.columns), tablefmt='pipe', floatfmt='.0f', showindex=False)))


In [ ]:
element_l = ['email','education', 'employment', 'works','funding']
input = 'Durst,Paul'
orcid = oneMatch_df[oneMatch_df['input']==input]['orcid'].values[0]
print(f'{input} has one ORCID match: {orcid}')
for element in element_l:
    data = orcid_dfs[element]
    df = data[data['orcid'] == orcid]
    if len(df) > 0:
        df = df.fillna('')
        display(Markdown(f'**{element}: ({len(df)})**'))
        display(Markdown(tabulate(df, headers=list(df.columns), tablefmt='pipe', floatfmt='.0f', showindex=False)))
    else:
        display(Markdown(f'{input} has no {element} metadata.'))

In [ ]:
author_df.to_csv('/Users/metadatagamechanger/ProjectsAndPlans/FieldStations/MooreaStudentPapers/studentORCIDs/matchedNames.csv', index=False)